# 🚚 LaneVoice — Carrier-Sales Voice AI (Colab)

A simple, open-source build of the PRD flow: **identify load → verify carrier →
negotiate (never below floor) → book or transfer**, with an audit-trail database.

* **Text demo** runs instantly — no models, **no API keys**.
* **Voice demo** uses free Hugging Face models (Whisper + Qwen2.5 + Kokoro) — still **no API keys**.
* **Real phone calls** use LiveKit + Twilio (scaffold + key list at the bottom).

> Tip: `Runtime → Change runtime type → GPU (T4)` for snappier voice. CPU also works.


## 1. Install (voice demo only — skip if you just want the text demo)

In [ ]:
# System dep for Kokoro TTS
!apt-get -qq install -y espeak-ng > /dev/null
# Free, open-source model stack (no keys)
!pip -q install faster-whisper transformers accelerate torch kokoro soundfile numpy
print("done")

## 2. Write the project files
Each cell writes one module so the notebook is self-contained.

In [ ]:
%%writefile database.py
"""
database.py
-----------
SQLite data layer for the Carrier Sales Voice AI agent.

This mirrors the core entities from the PRD (§10):
    Load, Carrier, Call, NegotiationOffer, TransferEvent

SQLite is used deliberately: it needs zero setup, lives in a single file, and
runs perfectly inside Google Colab. In production you would swap this module's
implementation for Postgres (per PRD §5.6) while keeping the same function
signatures.
"""

import sqlite3
import json
import datetime
from pathlib import Path

DB_PATH = Path(__file__).with_name("carrier_agent.db")


# --------------------------------------------------------------------------- #
# Schema
# --------------------------------------------------------------------------- #
SCHEMA = """
CREATE TABLE IF NOT EXISTS loads (
    load_id         TEXT PRIMARY KEY,
    origin          TEXT NOT NULL,
    destination     TEXT NOT NULL,
    pickup_date     TEXT NOT NULL,
    equipment       TEXT,
    weight_lbs      INTEGER,
    -- CARRIER-PAY economics. The carrier gets PAID; we (the broker) want to pay
    -- as little as possible. The agent OPENS low and walks its offer UP.
    open_rate       REAL NOT NULL,   -- agent's opening offer (starts here)
    ceiling_rate    REAL NOT NULL,   -- absolute max budget. Ask above this = NO DEAL, end call.
                                     -- Agent may only offer up to (ceiling - buffer); the
                                     -- buffer is reserved for a human to use on transfer.
    fraud_low_rate  REAL NOT NULL,   -- suspiciously cheap -> fraud review, don't just book
    assigned_rep_id TEXT,
    status          TEXT NOT NULL DEFAULT 'open'  -- open | covered | cancelled
);

CREATE TABLE IF NOT EXISTS carriers (
    mc_number             TEXT,
    usdot_number          TEXT PRIMARY KEY,   -- USDOT is primary per PRD §8.3
    legal_name            TEXT NOT NULL,
    authority_status      TEXT NOT NULL,      -- active | revoked | inactive
    insurance_on_file     INTEGER NOT NULL,   -- 1/0
    authority_reactivated_days INTEGER,       -- days since reactivation (fraud signal)
    last_verified_at      TEXT
);

CREATE TABLE IF NOT EXISTS reps (
    rep_id      TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    phone       TEXT NOT NULL,
    available   INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE IF NOT EXISTS calls (
    call_id      TEXT PRIMARY KEY,
    load_id      TEXT,
    carrier_dot  TEXT,
    start_time   TEXT,
    end_time     TEXT,
    outcome      TEXT,                -- booked | transferred | abandoned | rejected
    transcript   TEXT
);

CREATE TABLE IF NOT EXISTS negotiation_offers (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    call_id      TEXT,
    round_number INTEGER,
    offered_by   TEXT,                -- carrier | agent
    amount       REAL,
    timestamp    TEXT
);

CREATE TABLE IF NOT EXISTS transfer_events (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    call_id         TEXT,
    rep_id          TEXT,
    transfer_result TEXT,             -- connected | voicemail | failed
    timestamp       TEXT
);

CREATE TABLE IF NOT EXISTS call_notes (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    call_id    TEXT,
    note       TEXT,
    timestamp  TEXT
);
"""


def _now() -> str:
    return datetime.datetime.utcnow().isoformat()


def connect(db_path: Path = DB_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn


def init_db(db_path: Path = DB_PATH, seed: bool = True) -> None:
    """Create tables and (optionally) load sample data."""
    conn = connect(db_path)
    conn.executescript(SCHEMA)
    conn.commit()
    if seed:
        _seed(conn)
    conn.close()


def reset_db(db_path: Path = DB_PATH) -> None:
    """Drop the file and reseed — handy for repeatable demos/tests."""
    Path(db_path).unlink(missing_ok=True)
    init_db(db_path, seed=True)


# --------------------------------------------------------------------------- #
# Seed data  (stands in for a Transport Pro mirror — PRD §6)
# --------------------------------------------------------------------------- #
def _seed(conn: sqlite3.Connection) -> None:
    if conn.execute("SELECT COUNT(*) FROM loads").fetchone()[0] > 0:
        return  # already seeded

    loads = [
        # load_id, origin, dest, pickup, equip, weight,
        #   open(start), ceiling(hard max), fraud_low, rep, status
        #   agent's max offer = ceiling - 150 (buffer held for a human)
        ("L1001", "Chicago, IL", "Dallas, TX", "2026-07-25", "Dry Van", 42000,
         2000, 2500, 1400, "R01", "open"),   # agent can offer up to 2350
        ("L1002", "Atlanta, GA", "Miami, FL", "2026-07-24", "Reefer", 38000,
         1400, 1850, 1000, "R02", "open"),   # agent up to 1700
        ("L1003", "Los Angeles, CA", "Phoenix, AZ", "2026-07-26", "Flatbed", 45000,
         900, 1250, 650, "R01", "open"),      # agent up to 1100
        ("L1004", "Newark, NJ", "Boston, MA", "2026-07-23", "Dry Van", 30000,
         700, 950, 500, "R03", "covered"),    # agent up to 800
    ]
    conn.executemany(
        "INSERT INTO loads VALUES (?,?,?,?,?,?,?,?,?,?,?)", loads
    )

    carriers = [
        # mc, dot, name, authority, insured, reactivated_days, last_verified
        ("MC123456", "DOT1000001", "Blue Sky Logistics LLC", "active", 1, None, None),
        ("MC654321", "DOT2000002", "Roadrunner Freight Inc", "active", 1, None, None),
        ("MC999888", "DOT3000003", "Ghost Carrier LLC", "revoked", 0, None, None),
        ("MC777111", "DOT4000004", "Reactivated Haulers", "active", 1, 12, None),  # fraud signal
    ]
    conn.executemany(
        "INSERT INTO carriers VALUES (?,?,?,?,?,?,?)", carriers
    )

    reps = [
        ("R01", "Sarah Chen", "+15551110101", 1),
        ("R02", "Mike Torres", "+15551110102", 1),
        ("R03", "Priya Nair", "+15551110103", 0),  # unavailable -> tests fallback
    ]
    conn.executemany("INSERT INTO reps VALUES (?,?,?,?)", reps)
    conn.commit()


# --------------------------------------------------------------------------- #
# Read helpers
# --------------------------------------------------------------------------- #
def get_load(load_id: str, db_path: Path = DB_PATH):
    conn = connect(db_path)
    row = conn.execute(
        "SELECT * FROM loads WHERE UPPER(load_id)=UPPER(?)", (load_id,)
    ).fetchone()
    conn.close()
    return dict(row) if row else None


def get_open_loads(db_path: Path = DB_PATH):
    conn = connect(db_path)
    rows = conn.execute("SELECT * FROM loads WHERE status='open'").fetchall()
    conn.close()
    return [dict(r) for r in rows]


def get_carrier(mc_or_dot: str, db_path: Path = DB_PATH):
    """Look up by either MC or USDOT number (digits only comparison)."""
    q = "".join(ch for ch in mc_or_dot if ch.isdigit())
    conn = connect(db_path)
    row = conn.execute(
        """SELECT * FROM carriers
           WHERE REPLACE(REPLACE(mc_number,'MC',''),' ','')=?
              OR REPLACE(REPLACE(usdot_number,'DOT',''),' ','')=?""",
        (q, q),
    ).fetchone()
    conn.close()
    return dict(row) if row else None


def get_rep(rep_id: str, db_path: Path = DB_PATH):
    conn = connect(db_path)
    row = conn.execute("SELECT * FROM reps WHERE rep_id=?", (rep_id,)).fetchone()
    conn.close()
    return dict(row) if row else None


def get_available_rep_fallback(exclude_rep_id: str = None, db_path: Path = DB_PATH):
    conn = connect(db_path)
    row = conn.execute(
        "SELECT * FROM reps WHERE available=1 AND rep_id != ? LIMIT 1",
        (exclude_rep_id or "",),
    ).fetchone()
    conn.close()
    return dict(row) if row else None


# --------------------------------------------------------------------------- #
# Write helpers  (audit trail — PRD §9.4)
# --------------------------------------------------------------------------- #
def start_call(call_id: str, db_path: Path = DB_PATH):
    conn = connect(db_path)
    conn.execute(
        "INSERT OR REPLACE INTO calls (call_id, start_time) VALUES (?,?)",
        (call_id, _now()),
    )
    conn.commit()
    conn.close()


def log_offer(call_id, round_number, offered_by, amount, db_path: Path = DB_PATH):
    conn = connect(db_path)
    conn.execute(
        "INSERT INTO negotiation_offers (call_id, round_number, offered_by, amount, timestamp)"
        " VALUES (?,?,?,?,?)",
        (call_id, round_number, offered_by, amount, _now()),
    )
    conn.commit()
    conn.close()


def log_note(call_id, note, db_path: Path = DB_PATH):
    """Write a free-text note against the call (e.g. 'asked above ceiling')."""
    conn = connect(db_path)
    conn.execute(
        "INSERT INTO call_notes (call_id, note, timestamp) VALUES (?,?,?)",
        (call_id, note, _now()),
    )
    conn.commit()
    conn.close()


def log_transfer(call_id, rep_id, result, db_path: Path = DB_PATH):
    conn = connect(db_path)
    conn.execute(
        "INSERT INTO transfer_events (call_id, rep_id, transfer_result, timestamp)"
        " VALUES (?,?,?,?)",
        (call_id, rep_id, result, _now()),
    )
    conn.commit()
    conn.close()


def book_load(load_id: str, db_path: Path = DB_PATH):
    conn = connect(db_path)
    conn.execute("UPDATE loads SET status='covered' WHERE load_id=?", (load_id,))
    conn.commit()
    conn.close()


def end_call(call_id, load_id, carrier_dot, outcome, transcript, db_path: Path = DB_PATH):
    conn = connect(db_path)
    conn.execute(
        """UPDATE calls SET load_id=?, carrier_dot=?, end_time=?, outcome=?, transcript=?
           WHERE call_id=?""",
        (load_id, carrier_dot, _now(), outcome,
         json.dumps(transcript) if not isinstance(transcript, str) else transcript,
         call_id),
    )
    conn.commit()
    conn.close()


if __name__ == "__main__":
    init_db()
    print(f"Initialized {DB_PATH}")
    print("Open loads:", [l["load_id"] for l in get_open_loads()])


In [ ]:
%%writefile business_logic.py
"""
business_logic.py
-----------------
The deterministic "product" layer (PRD §4).

CORE SECURITY PRINCIPLE (PRD §4 / §9.4):
    The LLM is the conversational interface ONLY. It never decides whether an
    MC number is valid, and it never accepts a price. Every consequential
    decision is plain, unit-testable Python here. A caller cannot talk the
    model into a bad outcome because the model has no power to cause one.

Functions here are pure-ish (they read/write the DB) and return structured
dicts that the conversation layer turns into speech.
"""

import re
import database as db


# --------------------------------------------------------------------------- #
# Entity extraction from a spoken utterance (regex first — cheap & reliable)
# --------------------------------------------------------------------------- #
def extract_load_id(text: str):
    """Match things like 'L1001', 'load 1001', 'L 10 01'."""
    t = text.upper().replace(" ", "")
    m = re.search(r"L?\d{4,6}", t)
    if not m:
        return None
    digits = re.sub(r"\D", "", m.group())
    return f"L{digits}" if not m.group().startswith("L") else m.group()


def extract_mc_dot(text: str):
    """Return ('MC'|'DOT', number_str) or (None, None)."""
    t = text.upper()
    m = re.search(r"(MC|DOT|USDOT)[\s#:-]*(\d{4,8})", t)
    if m:
        kind = "DOT" if m.group(1) != "MC" else "MC"
        return kind, m.group(2)
    # bare number fallback
    m = re.search(r"\b(\d{6,8})\b", t)
    if m:
        return "DOT", m.group(1)
    return None, None


def extract_money(text: str):
    """Extract a dollar amount. Handles '$2,100', '2100', '21 hundred', '2.1k'."""
    t = text.lower().replace(",", "")
    m = re.search(r"\$?\s*(\d{3,6})(?:\s*(?:dollars|bucks))?", t)
    if m:
        return float(m.group(1))
    m = re.search(r"(\d+(?:\.\d+)?)\s*k", t)
    if m:
        return float(m.group(1)) * 1000
    return None


# --------------------------------------------------------------------------- #
# Step 2 — Load lookup
# --------------------------------------------------------------------------- #
def lookup_load(load_id: str) -> dict:
    load = db.get_load(load_id)
    if not load:
        return {"found": False}
    return {
        "found": True,
        "available": load["status"] == "open",
        "load": load,
    }


# --------------------------------------------------------------------------- #
# Step 3 — Carrier verification  (mock FMCSA / PRD §8)
# --------------------------------------------------------------------------- #
def verify_carrier(mc_or_dot: str) -> dict:
    """
    Deterministic verification with an explicit fraud-signal rule layer
    (PRD §8.2). In production, the DB lookup here is replaced by a call to
    FMCSA QCMobile + a commercial fallback, but the decision logic stays.
    """
    carrier = db.get_carrier(mc_or_dot)
    if not carrier:
        return {"verified": False, "reason": "not_found",
                "action": "human_review"}

    risk_flags = []
    if carrier["authority_status"] != "active":
        risk_flags.append(f"authority_{carrier['authority_status']}")
    if not carrier["insurance_on_file"]:
        risk_flags.append("insurance_lapse")
    if carrier["authority_reactivated_days"] is not None and \
            carrier["authority_reactivated_days"] <= 90:
        risk_flags.append("recently_reactivated")

    hard_fail = carrier["authority_status"] != "active" or not carrier["insurance_on_file"]

    return {
        "verified": not hard_fail,
        "high_risk": bool(risk_flags),
        "risk_flags": risk_flags,
        "carrier": carrier,
        # PRD §3 step 3: fraud attempts are logged & routed to review, never dropped
        "action": "proceed" if not hard_fail and not risk_flags
                  else "human_review",
    }


# --------------------------------------------------------------------------- #
# Step 5 — Negotiation engine  (THE hard server-side check — PRD §9.4)
# --------------------------------------------------------------------------- #
class Negotiation:
    """
    Deterministic negotiation state for one call/load.

    Strategy (as specified by the desk):
      * The agent OPENS at `open_rate` (a low starting offer).
      * If the carrier wants more, the agent walks its offer UP by `step`
        (~25-30, larger when the gap is big).
      * The agent may NEVER offer more than  ceiling - BUFFER  (BUFFER=150).
        That $150 is held back for a human to use on a warm transfer.
      * If the carrier asks ABOVE the ceiling -> NO DEAL: log a note, decline,
        end the call.

    accept/reject/no-deal is decided ONLY here, against the live ceiling — the
    LLM cannot influence it (PRD §9.4).
    """

    BUFFER = 150          # $ held below ceiling; agent may not cross it
    STEP_SMALL = 25       # normal increment
    STEP_BIG = 30         # increment when the carrier is still far away

    def __init__(self, load: dict, max_rounds: int = 6):
        self.load = load
        self.open = load["open_rate"]              # advertised/opening (low) price
        self.ceiling = load["ceiling_rate"]        # true budget max
        self.fraud_low = load["fraud_low_rate"]
        self.agent_max = max(self.open, self.ceiling - self.BUFFER)  # most agent can offer
        self.max_rounds = max_rounds
        self.round = 0
        self.last_agent_offer = self.open          # current offer on the table
        self.held_firm = False                     # have we pushed back once yet?
        self.history = []

    def _step(self, gap: float) -> int:
        return self.STEP_BIG if gap > 100 else self.STEP_SMALL

    def evaluate(self, carrier_ask: float) -> dict:
        """
        `carrier_ask` = the rate the carrier wants to be PAID.

        Behaves like a human rep: on a high ask it HOLDS FIRM once (restates the
        opening), then WALKS UP by 25-30 per round, and only disconnects (with a
        clear note) once it has reached its cap or run out of patience. It never
        hangs up on the first high ask.

        Returns one of:
          {'decision': 'accept',  'rate': X}
          {'decision': 'hold',    'rate': opening}           # restate low price, don't move
          {'decision': 'counter', 'rate': X}                 # walked our offer up
          {'decision': 'review',  'reason': 'suspiciously_low'}
          {'decision': 'no_deal', 'reason': 'stalemate', 'ask': X,
           'final_offer': X, 'within_ceiling': bool}         # disconnect + note
        """
        self.round += 1
        self.history.append(("carrier", carrier_ask))

        # Fraud tripwire: absurdly cheap -> double-brokering / no-show risk.
        if carrier_ask < self.fraud_low:
            return {"decision": "review", "reason": "suspiciously_low"}

        # Carrier is at/below what we're already offering -> book it (cheap for us).
        if carrier_ask <= self.last_agent_offer:
            return {"decision": "accept", "rate": carrier_ask}

        # Carrier wants MORE than our current offer.
        # First push-back: hold firm and restate the opening price (don't move yet).
        if not self.held_firm:
            self.held_firm = True
            return {"decision": "hold", "rate": self.last_agent_offer,
                    "ask": carrier_ask}

        # Already held once -> consider walking our offer UP toward the cap.
        gap = carrier_ask - self.last_agent_offer
        proposed = min(self.last_agent_offer + self._step(gap), self.agent_max)
        moved = proposed > self.last_agent_offer

        # If we still can't reach them and either can't move further (at cap) or
        # patience is spent, walk away at our LAST OFFER ACTUALLY MADE.
        if carrier_ask > proposed and (not moved
                                       or proposed >= self.agent_max
                                       or self.round >= self.max_rounds):
            return {"decision": "no_deal", "reason": "stalemate",
                    "ask": carrier_ask, "final_offer": self.last_agent_offer,
                    "within_ceiling": carrier_ask <= self.ceiling}

        # Make the raised offer.
        self.last_agent_offer = proposed
        self.history.append(("agent", proposed))
        if carrier_ask <= proposed:            # our offer now meets their ask
            return {"decision": "accept", "rate": carrier_ask}
        return {"decision": "counter", "rate": proposed}


# --------------------------------------------------------------------------- #
# Step 6b — Rep lookup + transfer  (PRD §3 / §9.5)
# --------------------------------------------------------------------------- #
def resolve_transfer(load: dict) -> dict:
    rep = db.get_rep(load["assigned_rep_id"]) if load.get("assigned_rep_id") else None
    if rep and rep["available"]:
        return {"transfer_to": rep, "fallback": False}
    # PRD §9.5: never dead-air disconnect — fall back to another available rep
    fallback = db.get_available_rep_fallback(
        exclude_rep_id=load.get("assigned_rep_id"))
    if fallback:
        return {"transfer_to": fallback, "fallback": True}
    return {"transfer_to": None, "fallback": True,
            "note": "voicemail_plus_callback_task"}


In [ ]:
%%writefile conversation.py
"""
conversation.py
---------------
The call state machine (PRD §3). This is the agent's "brain", independent of
how audio gets in/out. It drives:

    GREETING -> IDENTIFY_LOAD -> VERIFY_CARRIER -> STATE_PRICE
             -> NEGOTIATE -> RESOLVE (book | transfer | abandon)

The LLM is used ONLY to phrase the reply naturally given the state + the facts
that deterministic code (business_logic.py) has already decided. If no LLM is
provided, it falls back to clean template strings — so the whole flow is
testable with zero models loaded.
"""

import uuid
import business_logic as bl
import database as db

CONSENT_LINE = ("Thanks for calling the load desk. Just so you know, this call "
                "may be recorded for quality purposes. ")


class CarrierSalesAgent:
    def __init__(self, llm=None, max_rounds: int = 6):
        self.llm = llm                 # optional: object with .phrase(instruction, context)
        self.max_rounds = max_rounds
        self.call_id = f"CALL-{uuid.uuid4().hex[:8]}"
        self.state = "GREETING"
        self.load = None
        self.carrier = None
        self.neg = None
        self.transcript = []
        self.outcome = None
        db.start_call(self.call_id)

    # -- phrasing helper ---------------------------------------------------- #
    def _say(self, fallback: str, instruction: str = None, context: str = None):
        """Return agent speech. Use LLM to naturalize if available."""
        text = fallback
        if self.llm is not None and instruction is not None:
            try:
                text = self.llm.phrase(instruction, context or "")
            except Exception:
                text = fallback
        self.transcript.append(("agent", text))
        return text

    def _log_user(self, text: str):
        self.transcript.append(("carrier", text))

    # -- main entry --------------------------------------------------------- #
    def greeting(self) -> str:
        self.state = "IDENTIFY_LOAD"
        return self._say(
            CONSENT_LINE + "Which load are you calling about? "
            "You can give me the load ID or the lane.",
            instruction="Greet a carrier calling about a freight load. Include a "
                        "one-line call-recording disclosure, then ask which load "
                        "they want (load ID or lane). Be brief and professional.",
        )

    def handle(self, user_text: str) -> str:
        """Feed one carrier utterance, get the agent's spoken reply."""
        self._log_user(user_text)
        handler = {
            "IDENTIFY_LOAD": self._identify_load,
            "VERIFY_CARRIER": self._verify_carrier,
            "STATE_PRICE": self._negotiate,   # first offer arrives here
            "NEGOTIATE": self._negotiate,
            "RESOLVE": self._resolve_followup,
            "DONE": lambda t: "This call has ended. Goodbye.",
        }.get(self.state, self._identify_load)
        return handler(user_text)

    # -- Step 2: identify load --------------------------------------------- #
    def _identify_load(self, text: str) -> str:
        load_id = bl.extract_load_id(text)
        if not load_id:
            return self._say(
                "I didn't catch a load ID. Could you repeat it? For example, L1001.",
                instruction="Politely say you didn't catch the load ID and ask them "
                            "to repeat it, giving 'L1001' as an example format.")
        result = bl.lookup_load(load_id)
        if not result["found"]:
            opens = ", ".join(l["load_id"] for l in db.get_open_loads())
            return self._say(
                f"I couldn't find load {load_id}. Open loads right now are {opens}. "
                "Which one would you like?",
                instruction=f"Tell the caller load {load_id} was not found, then offer "
                            f"the open loads: {opens}. Ask which they want.")
        if not result["available"]:
            opens = ", ".join(l["load_id"] for l in db.get_open_loads())
            return self._say(
                f"Sorry, load {load_id} is already covered. Other open loads: {opens}.",
                instruction=f"Tell the caller load {load_id} is already covered and "
                            f"offer open loads: {opens}.")

        self.load = result["load"]
        self.state = "VERIFY_CARRIER"
        l = self.load
        return self._say(
            f"Got it — load {l['load_id']}, {l['origin']} to {l['destination']}, "
            f"picking up {l['pickup_date']}, {l['equipment']}. "
            "To move forward I'll need your MC or USDOT number.",
            instruction="Confirm the load back to the carrier, then ask for their MC "
                        "or USDOT number.",
            context=f"Load: {l['load_id']} {l['origin']}->{l['destination']} "
                    f"pickup {l['pickup_date']} equip {l['equipment']}")

    # -- Step 3: verify carrier -------------------------------------------- #
    def _verify_carrier(self, text: str) -> str:
        kind, number = bl.extract_mc_dot(text)
        if not number:
            return self._say(
                "I didn't get that. Please say your MC or USDOT number slowly.",
                instruction="Say you didn't catch the number and ask them to repeat "
                            "their MC or USDOT number slowly.")
        v = bl.verify_carrier(number)
        if not v["verified"]:
            self.state = "DONE"
            self.outcome = "rejected"
            self._finish("rejected")
            reason = v.get("reason", "risk flags")
            return self._say(
                "I'm not able to verify active authority and insurance on that number, "
                "so I can't discuss rate right now. I'm routing this to our team for "
                "review and someone will follow up. Thanks for calling.",
                instruction="Firmly but politely explain you cannot verify active "
                            "authority/insurance so you cannot discuss rate, and that "
                            "it is being routed to a human for review. Do not reveal "
                            "internal fraud logic.")
        if v["high_risk"]:
            # verified but flagged -> human review, still logged (never dropped)
            self.carrier = v["carrier"]
            self.state = "DONE"
            self.outcome = "transferred"
            rt = bl.resolve_transfer(self.load)
            self._finish("transferred", rep=rt.get("transfer_to"))
            return self._say(
                "Thanks. I want to get you to a rep directly to finish verification — "
                "one moment while I connect you.",
                instruction="Tell the carrier you're connecting them to a human rep to "
                            "finish verification. Keep it smooth and non-accusatory.")

        self.carrier = v["carrier"]
        self.neg = bl.Negotiation(self.load, max_rounds=self.max_rounds)
        self.state = "STATE_PRICE"
        opening = int(self.neg.open)
        db.log_offer(self.call_id, 0, "agent", opening)
        return self._say(
            f"You're verified, {self.carrier['legal_name']}. "
            f"I've got this one at ${opening}. Does that work for you?",
            instruction=f"Confirm the carrier is verified, then OFFER them ${opening} "
                        "for the load and ask if it works. This is an opening offer.",
            context=f"Carrier: {self.carrier['legal_name']}. Opening offer ${opening}.")

    # -- Steps 4/5: negotiate ---------------------------------------------- #
    def _negotiate(self, text: str) -> str:
        # accept detection (carrier agrees to the rate on the table)
        low = text.lower()
        accept_words = ["that works", "that'll work", "works for me", "deal",
                        "i'll take it", "sounds good", "book it", "agreed",
                        "accept", "perfect", "yes", "yeah", "yep", "yup",
                        "ok", "okay", "sure", "fine"]
        if any(w in low for w in accept_words) and bl.extract_money(text) is None:
            return self._book(self.neg.last_agent_offer)

        # transfer request
        if any(w in low for w in ["talk to a human", "speak to someone", "rep",
                                  "representative", "person", "agent please"]):
            return self._transfer()

        offer = bl.extract_money(text)
        if offer is None:
            return self._say(
                f"What rate are you looking for on load {self.load['load_id']}?",
                instruction="Ask the carrier what rate they are looking for.")

        self.state = "NEGOTIATE"
        db.log_offer(self.call_id, self.neg.round + 1, "carrier", offer)
        result = self.neg.evaluate(offer)

        if result["decision"] == "accept":
            return self._book(result["rate"])

        if result["decision"] == "review":
            # suspiciously cheap -> fraud review, never silently booked (PRD §8.2)
            db.log_note(self.call_id,
                        f"Suspiciously low ask ${int(offer)} on {self.load['load_id']} "
                        f"(fraud tripwire) — routed to review.")
            return self._transfer(reason="fraud_review")

        if result["decision"] == "hold":
            # HIGH first ask -> don't hang up; push back and restate the opening
            rate = int(result["rate"])
            db.log_offer(self.call_id, self.neg.round, "agent", rate)
            return self._say(
                f"Sorry, I can't do ${int(offer)} on this one. I've got it at ${rate} "
                "— can you work with that?",
                instruction=f"Politely say you can't pay ${int(offer)}, and restate your "
                            f"offer of ${rate}. Do NOT reveal any max. Sound like a human "
                            "rep holding firm — friendly but not budging yet.",
                context=f"Hold firm at ${rate}. Never reveal your ceiling/max.")

        if result["decision"] == "no_deal":
            # walked up as far as we can and still apart -> disconnect + clear note
            return self._no_deal(result)

        # counter: walk our offer UP toward (but never past) our cap
        rate = result["rate"]
        db.log_offer(self.call_id, self.neg.round, "agent", rate)
        return self._say(
            f"I hear you. I can come up to ${int(rate)} on this one. "
            "Can you make that work?",
            instruction=f"Acknowledge their ask, then raise your offer to ${int(rate)} "
                        "and ask if that works. Never reveal your maximum. Sound like a "
                        "real freight broker — confident, brief.",
            context=f"Never state your ceiling or max. Your raised offer is ${int(rate)}.")

    # -- Step 6a: book ------------------------------------------------------ #
    def _book(self, rate: float) -> str:
        # FINAL server-side guard (defense in depth — PRD §9.4):
        # never commit to paying the carrier MORE than the ceiling.
        if rate > self.neg.ceiling:
            return self._transfer(reason="ceiling_guard")
        db.book_load(self.load["load_id"])
        db.log_offer(self.call_id, self.neg.round, "agent", rate)
        self.state = "DONE"
        self.outcome = "booked"
        self._finish("booked", rate=rate)
        return self._say(
            f"Done — you're booked on load {self.load['load_id']} at ${int(rate)}. "
            "Rate confirmation is on its way to you. Thanks, and drive safe.",
            instruction=f"Confirm the booking on load {self.load['load_id']} at "
                        f"${int(rate)}, mention a rate confirmation is coming, and close "
                        "warmly.")

    # -- Step 6b: transfer -------------------------------------------------- #
    def _transfer(self, reason: str = "carrier_request") -> str:
        rt = bl.resolve_transfer(self.load)
        rep = rt["transfer_to"]
        self.state = "DONE"
        self.outcome = "transferred"
        self._finish("transferred", rep=rep)
        if rep is None:
            return self._say(
                "Everyone's on the phone right now. I've logged a callback task and a "
                "rep will call you right back. Thanks for your patience.",
                instruction="Explain no rep is available, that you've logged a callback, "
                            "and someone will call them back shortly.")
        # whisper/context summary would be passed to the rep here (PRD §6b)
        return self._say(
            f"Let me connect you with {rep['name']}, who handles this load. "
            "One moment.",
            instruction=f"Tell the carrier you're connecting them to {rep['name']} who "
                        "handles this load.")

    # -- No deal: negotiated, still apart -> clear note + decline + end call -- #
    def _no_deal(self, result: dict) -> str:
        ask = int(result["ask"])
        final = int(result["final_offer"])
        # A clear, human-readable note capturing the whole negotiation (PRD §9.4).
        offers = [f"${int(a)}" for who, a in self.neg.history if who == "agent"]
        followup = ("Carrier's number is within our ceiling — a rep could still "
                    "close it using the reserved buffer."
                    if result.get("within_ceiling")
                    else "Carrier's number is above our ceiling — not workable.")
        db.log_note(
            self.call_id,
            f"NO DEAL on {self.load['load_id']} ({self.load['origin']} -> "
            f"{self.load['destination']}). Carrier {self.carrier['legal_name']} "
            f"(USDOT {self.carrier['usdot_number']}) held at ${ask}. We opened at "
            f"${int(self.neg.open)} and walked up to ${final} "
            f"(offers: {', '.join(offers) or 'n/a'}) over {self.neg.round} rounds; "
            f"no agreement. {followup} Call ended by agent.")
        self.state = "DONE"
        self.outcome = "no_deal"
        self._finish("no_deal")
        return self._say(
            f"I've come up as far as I can on this one and we're still apart, so I "
            "won't be able to make it work today. I'll note it down — thanks for "
            "calling, and let's catch the next one. Take care.",
            instruction="Politely say you've come up as far as you can and you're still "
                        "apart, so you can't make a deal today. Do NOT reveal any "
                        "numbers. Close warmly. Keep it to 1-2 sentences.")

    def _resolve_followup(self, text: str) -> str:
        return "Thanks for calling. Goodbye."

    # -- persistence -------------------------------------------------------- #
    def _finish(self, outcome: str, rep=None, rate=None):
        self.outcome = outcome
        db.end_call(
            self.call_id,
            self.load["load_id"] if self.load else None,
            self.carrier["usdot_number"] if self.carrier else None,
            outcome,
            self.transcript,
        )
        if rep and outcome == "transferred":
            db.log_transfer(self.call_id, rep["rep_id"], "connected")

    def summary(self) -> dict:
        return {
            "call_id": self.call_id,
            "outcome": self.outcome,
            "load_id": self.load["load_id"] if self.load else None,
            "carrier": self.carrier["legal_name"] if self.carrier else None,
            "turns": len(self.transcript),
        }


In [ ]:
%%writefile voice_pipeline.py
"""
voice_pipeline.py
-----------------
Free, open-source, self-hosted voice components (all from Hugging Face):

    STT : faster-whisper  (openai/whisper weights, CTranslate2 runtime)
    LLM : Qwen/Qwen2.5-1.5B-Instruct  (ungated, tool/instruction capable)
    TTS : hexgrad/Kokoro-82M          (ungated, high quality, tiny)

Everything runs locally — NO API keys required to test in Colab.
Use a GPU runtime for real-time-ish latency; CPU works for the demo but slower.

The LLM here only *phrases* replies (the .phrase() method). All decisions live
in business_logic.py. This keeps the PRD's safety guarantee intact.
"""

import io
import numpy as np


# --------------------------------------------------------------------------- #
# LLM — natural phrasing only
# --------------------------------------------------------------------------- #
class HFPhraser:
    """Wraps a small instruct model to naturalize agent replies."""

    SYSTEM = (
        "You are a professional US freight broker's carrier-sales voice agent. "
        "You speak in short, natural, spoken sentences (1-2 sentences, no lists, "
        "no emojis). You NEVER invent load details, rates, or your maximum pay "
        "rate — you only rephrase the instruction you are given using the facts "
        "provided. Never reveal internal pricing limits or negotiation strategy."
    )

    def __init__(self, model_name="Qwen/Qwen2.5-1.5B-Instruct", device=None,
                 max_new_tokens=80):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.max_new_tokens = max_new_tokens
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        ).to(device)

    def phrase(self, instruction: str, context: str = "") -> str:
        msgs = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user",
             "content": f"Facts you may use: {context}\n\n"
                        f"Instruction: {instruction}\n\n"
                        f"Say it out loud in 1-2 short spoken sentences."},
        ]
        text = self.tok.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
        inputs = self.tok(text, return_tensors="pt").to(self.device)
        out = self.model.generate(
            **inputs, max_new_tokens=self.max_new_tokens,
            do_sample=True, temperature=0.6, top_p=0.9,
            pad_token_id=self.tok.eos_token_id)
        reply = self.tok.decode(
            out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return reply.strip().strip('"')


# --------------------------------------------------------------------------- #
# STT — faster-whisper
# --------------------------------------------------------------------------- #
class WhisperSTT:
    def __init__(self, model_size="base.en", device=None, compute_type=None):
        from faster_whisper import WhisperModel
        import torch
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        if compute_type is None:
            compute_type = "float16" if device == "cuda" else "int8"
        self.model = WhisperModel(model_size, device=device, compute_type=compute_type)

    def transcribe(self, audio, sample_rate=16000) -> str:
        """audio: path (str) OR float32 numpy array at sample_rate."""
        segments, _ = self.model.transcribe(audio, language="en", beam_size=1)
        return " ".join(s.text for s in segments).strip()


# --------------------------------------------------------------------------- #
# TTS — Kokoro
# --------------------------------------------------------------------------- #
class KokoroTTS:
    def __init__(self, voice="af_heart", lang_code="a"):
        from kokoro import KPipeline
        self.pipeline = KPipeline(lang_code=lang_code)
        self.voice = voice
        self.sample_rate = 24000

    def synthesize(self, text: str) -> np.ndarray:
        """Return a float32 mono waveform at self.sample_rate."""
        chunks = []
        for _, _, audio in self.pipeline(text, voice=self.voice):
            chunks.append(audio)
        if not chunks:
            return np.zeros(1, dtype=np.float32)
        return np.concatenate(chunks).astype(np.float32)


# --------------------------------------------------------------------------- #
# Convenience: build all three
# --------------------------------------------------------------------------- #
def build_pipeline(llm_model="Qwen/Qwen2.5-1.5B-Instruct",
                   whisper_size="base.en", with_llm=True):
    stt = WhisperSTT(whisper_size)
    tts = KokoroTTS()
    llm = HFPhraser(llm_model) if with_llm else None
    return stt, llm, tts


In [ ]:
%%writefile run_demo.py
"""
run_demo.py
-----------
Text-mode simulation of the carrier-sales flow. Uses NO models and NO API keys
-- pure business logic + state machine. Great for verifying the negotiation /
verification / transfer logic before you add voice.

    python run_demo.py            # scripted demo calls
    python run_demo.py --chat     # interactive: you play the carrier
"""

import sys
import database as db
from conversation import CarrierSalesAgent


def scripted(name, turns, max_rounds=6):
    print(f"\n{'='*70}\nSCENARIO: {name}\n{'='*70}")
    db.reset_db()                                # fresh load data each scenario
    agent = CarrierSalesAgent(llm=None, max_rounds=max_rounds)  # template phrasing
    print(f"AGENT : {agent.greeting()}")
    for t in turns:
        print(f"CALLER: {t}")
        print(f"AGENT : {agent.handle(t)}")
        if agent.state == "DONE":
            break
    print(f"--> outcome: {agent.summary()}")


def interactive():
    agent = CarrierSalesAgent(llm=None)
    print(f"AGENT : {agent.greeting()}")
    while agent.state != "DONE":
        try:
            t = input("CALLER: ")
        except EOFError:
            break
        print(f"AGENT : {agent.handle(t)}")
    print("Summary:", agent.summary())


if __name__ == "__main__":
    db.init_db()
    if "--chat" in sys.argv:
        interactive()
    else:
        # L1001: opens 2000, ceiling 2500 (agent may offer up to 2350)
        scripted("Carrier accepts the opening offer", [
            "Hi, I'm calling about load L1001",
            "My MC number is 123456",
            "yeah that works",            # accepts our opening $2000
        ])
        scripted("Agent walks its offer UP by 25-30, carrier then accepts", [
            "L1001",
            "MC 123456",
            "I need 2080 for that",       # >2000 -> agent comes up to 2025
            "come on, 2060",              # -> agent comes up to 2050
            "ok deal",                    # books at agent's $2050 offer
        ])
        # High first ask -> HOLD firm, then WALK UP, then disconnect + note
        # (never hangs up on the first attempt — behaves like a human rep)
        scripted("High ask -> hold firm -> walk up -> disconnect + clear note", [
            "load 1003",
            "MC654321",
            "I need 1500 for this",       # >> opening 900 -> HOLD, restate 900
            "no way, 1500",               # still apart -> walk up to 930
            "come on, 1500",              # -> walk up to 960
            "1500 or nothing",            # patience spent -> disconnect + note
        ], max_rounds=4)
        scripted("Suspiciously cheap -> fraud review, not auto-booked", [
            "L1001",
            "MC123456",
            "I'll haul it for 900",       # < fraud_low 1400 -> human review
        ])
        scripted("Revoked authority -> human review, not hung up", [
            "L1002",
            "MC999888",                   # revoked
        ])


## 3. Text demo — the whole flow, no models, no keys
This proves the deterministic logic: booking, floor enforcement, verification, transfer.

In [ ]:
import importlib, database, business_logic, conversation, run_demo
for m in (database, business_logic, conversation, run_demo):
    importlib.reload(m)
run_demo.scripted("Accept opening offer", ["about L1001", "MC 123456", "yeah that works"])
run_demo.scripted("Agent walks offer up, then carrier accepts",
                  ["L1001", "MC 123456", "I need 2080", "come on 2060", "ok deal"])
run_demo.scripted("High ask -> hold firm -> walk up -> disconnect + note",
                  ["load 1003", "MC654321", "I need 1500", "no way 1500",
                   "come on 1500", "1500 or nothing"], max_rounds=4)

## 4. Interactive text chat (you play the carrier)
Run, then type carrier lines. Try: `L1001` → `MC 123456` → `yes` (books at the
opening $2000); or ask `2080`, then `2060`, then `deal` to watch it walk up.

In [ ]:
from conversation import CarrierSalesAgent
import database; database.reset_db()
agent = CarrierSalesAgent(llm=None)
print("AGENT:", agent.greeting())
while agent.state != "DONE":
    t = input("YOU (carrier): ")
    print("AGENT:", agent.handle(t))
print("SUMMARY:", agent.summary())

## 5. Voice demo 🔊 (loads free HF models)
First run downloads weights (~1–2 GB). The agent's reply is *spoken* by Kokoro
and phrased by Qwen2.5. Decisions still come from the deterministic engine.

In [ ]:
from voice_pipeline import build_pipeline
# with_llm=True uses Qwen2.5 to naturalize phrasing; set False to skip the LLM.
stt, llm, tts = build_pipeline(with_llm=True)
print("models loaded")

In [ ]:
from conversation import CarrierSalesAgent
from IPython.display import Audio, display
import database; database.reset_db()

agent = CarrierSalesAgent(llm=llm)     # LLM phrases, engine decides
def speak(text):
    print("AGENT:", text)
    wav = tts.synthesize(text)
    display(Audio(wav, rate=tts.sample_rate, autoplay=False))

speak(agent.greeting())

Type a carrier line; hear the spoken reply. Re-run the cell for each turn.

In [ ]:
carrier_line = "I'm calling about load L1001"   #@param {type:"string"}
reply = agent.handle(carrier_line)
speak(reply)
print("state:", agent.state)

### (Optional) Real speech-to-text
Upload a short WAV of you speaking a carrier line — Whisper transcribes it, the
agent responds by voice.

In [ ]:
from google.colab import files
import soundfile as sf, numpy as np
up = files.upload()                      # choose a .wav (mono, any rate)
path = list(up.keys())[0]
audio, sr = sf.read(path)
if audio.ndim > 1: audio = audio.mean(axis=1)
audio = audio.astype("float32")
text = stt.transcribe(audio, sr)
print("HEARD:", text)
speak(agent.handle(text))

## 6. Inspect the audit trail (database)
Every call, offer, and transfer is logged (PRD §9.4).

In [ ]:
import database as db, pandas as pd
conn = db.connect()
for t in ["calls", "negotiation_offers", "transfer_events", "call_notes", "loads"]:
    print(f"\n=== {t} ===")
    display(pd.read_sql_query(f"SELECT * FROM {t}", conn))
conn.close()

## 7. Going live with LiveKit + Twilio (deployment)

Real inbound calls **don't run in Colab** (no stable public endpoint). Deploy the
worker below on a small **GPU host** — it dials *out* to LiveKit Cloud, so no
inbound ports are needed.

**API keys you'll need (none required for the demos above):**

| Service | Keys | Purpose |
|---|---|---|
| LiveKit (free tier) | `LIVEKIT_URL`, `LIVEKIT_API_KEY`, `LIVEKIT_API_SECRET` | media + SIP bridge |
| Twilio | `TWILIO_ACCOUNT_SID`, `TWILIO_AUTH_TOKEN`, a Voice number + Elastic SIP Trunk | the phone number carriers dial |
| Hugging Face (optional) | `HF_TOKEN` | only for *gated* models (Llama etc.) |
| FMCSA QCMobile (optional) | `FMCSA_WEBKEY` | real carrier authority/insurance verification |

The worker file is written below; run it on your GPU host, not here.

In [ ]:
%%writefile livekit_agent.py
"""
livekit_agent.py
----------------
DEPLOYMENT SCAFFOLD — connects the same brain (conversation.py) to real phone
calls via LiveKit Agents + Twilio SIP.

This is NOT meant to run inside Colab for live inbound calls (Colab has no
stable public endpoint and sessions time out). Run it on a small GPU host
(the LiveKit worker connects OUT to LiveKit Cloud, so it needs no inbound
ports of its own).

Pipeline:  Twilio PSTN  ->  Twilio SIP trunk  ->  LiveKit SIP  ->  this worker
           worker:  Silero VAD -> Whisper STT -> (our state machine) -> Kokoro TTS

Install (on the GPU host, Python 3.11+):
    pip install "livekit-agents>=1.0" livekit-plugins-silero \
                faster-whisper kokoro soundfile transformers torch

Env vars required (see README "API keys"):
    LIVEKIT_URL, LIVEKIT_API_KEY, LIVEKIT_API_SECRET

Run:
    python livekit_agent.py dev        # local dev
    python livekit_agent.py start      # production worker
"""

import asyncio
import numpy as np

from livekit import rtc
from livekit.agents import (
    Agent, AgentSession, JobContext, WorkerOptions, cli,
    stt as lk_stt, tts as lk_tts,
)
from livekit.plugins import silero

from voice_pipeline import WhisperSTT, KokoroTTS, HFPhraser
from conversation import CarrierSalesAgent


# --------------------------------------------------------------------------- #
# Thin adapters wrapping our HF models in LiveKit's STT/TTS interfaces.
# (LiveKit's plugin API evolves; verify method names against your installed
#  livekit-agents version. These follow the v1.x streaming conventions.)
# --------------------------------------------------------------------------- #
class LocalWhisperSTT(lk_stt.STT):
    def __init__(self):
        super().__init__(capabilities=lk_stt.STTCapabilities(
            streaming=False, interim_results=False))
        self._model = WhisperSTT("base.en")

    async def _recognize_impl(self, buffer, *, language=None, conn_options=None):
        frame = rtc.combine_audio_frames(buffer)
        samples = np.frombuffer(frame.data, dtype=np.int16).astype(np.float32) / 32768.0
        # resample to 16k if needed (Whisper wants 16k)
        text = await asyncio.to_thread(self._model.transcribe, samples, 16000)
        return lk_stt.SpeechEvent(
            type=lk_stt.SpeechEventType.FINAL_TRANSCRIPT,
            alternatives=[lk_stt.SpeechData(text=text, language="en")],
        )


class LocalKokoroTTS(lk_tts.TTS):
    def __init__(self):
        self._model = KokoroTTS()
        super().__init__(
            capabilities=lk_tts.TTSCapabilities(streaming=False),
            sample_rate=self._model.sample_rate, num_channels=1)

    def synthesize(self, text, *, conn_options=None):
        return _KokoroStream(self, text, self._model)


class _KokoroStream(lk_tts.ChunkedStream):
    def __init__(self, tts, text, model):
        super().__init__(tts=tts, input_text=text)
        self._model = model

    async def _run(self, output_emitter):
        wav = await asyncio.to_thread(self._model.synthesize, self.input_text)
        pcm16 = (np.clip(wav, -1, 1) * 32767).astype(np.int16).tobytes()
        output_emitter.initialize(
            request_id="kokoro", sample_rate=self._model.sample_rate,
            num_channels=1, mime_type="audio/pcm")
        output_emitter.push(pcm16)
        output_emitter.flush()


# --------------------------------------------------------------------------- #
# The agent: bridges LiveKit's turn events into our deterministic brain.
# --------------------------------------------------------------------------- #
class CarrierAgent(Agent):
    def __init__(self):
        super().__init__(instructions="Carrier sales agent (logic in conversation.py).")
        self.brain = CarrierSalesAgent(llm=HFPhraser())

    async def on_enter(self):
        await self.session.say(self.brain.greeting())

    async def on_user_turn_completed(self, turn_ctx, new_message):
        user_text = new_message.text_content or ""
        reply = await asyncio.to_thread(self.brain.handle, user_text)
        await self.session.say(reply)
        if self.brain.state == "DONE":
            await asyncio.sleep(1)
            await self.session.aclose()


async def entrypoint(ctx: JobContext):
    await ctx.connect()
    session = AgentSession(
        vad=silero.VAD.load(),
        stt=LocalWhisperSTT(),
        tts=LocalKokoroTTS(),
        # llm handled inside CarrierAgent (our state machine), not a chat LLM node
    )
    await session.start(agent=CarrierAgent(), room=ctx.room)


if __name__ == "__main__":
    cli.run_app(WorkerOptions(entrypoint_fnc=entrypoint))


# --------------------------------------------------------------------------- #
# TWILIO  ->  LIVEKIT SIP  wiring (one-time setup, done via CLI/console)
# --------------------------------------------------------------------------- #
"""
1. Buy a Twilio phone number (Voice-capable).

2. Create a Twilio Elastic SIP Trunk:
     - Termination URI: <your-name>.pstn.twilio.com
     - Origination:  point to LiveKit SIP URI  (sip:<project>.sip.livekit.cloud)
     - Assign your Twilio number to the trunk.

3. In LiveKit, create an inbound SIP trunk + dispatch rule so incoming calls
   are routed to a room your worker joins. Using the LiveKit CLI:

     lk sip inbound create --file inbound-trunk.json
     lk sip dispatch create --file dispatch-rule.json

   inbound-trunk.json:
     { "trunk": { "name": "twilio-in", "numbers": ["+1XXXXXXXXXX"] } }

   dispatch-rule.json:
     { "dispatch_rule": {
         "rule": { "dispatchRuleIndividual": { "roomPrefix": "call-" } } } }

4. Deploy this worker on a GPU host and run:  python livekit_agent.py start
   The worker dials OUT to LIVEKIT_URL, so no inbound firewall ports needed.

5. Call your Twilio number -> Twilio -> LiveKit -> this worker answers.
"""


In [ ]:
# On your GPU host (NOT Colab):
#   pip install "livekit-agents>=1.0" livekit-plugins-silero faster-whisper kokoro soundfile transformers torch
#   export LIVEKIT_URL=... LIVEKIT_API_KEY=... LIVEKIT_API_SECRET=...
#   python livekit_agent.py start
print("See the Twilio->LiveKit SIP setup steps at the bottom of livekit_agent.py")